In [ ]:
%pip install lightgbm
%reload_ext autoreload

In [ ]:
import joblib
import re
import os
import pandas as pd
import mlflow
import mlflow.lightgbm
from mlflow.tracking import MlflowClient

MODELS_VOLUME = "/Volumes/workspace/default/models"
METRICS_CSV   = f"{MODELS_VOLUME}/08_train_regional_models_with_weather.csv"
# Set via env var EV_MLFLOW_EXPERIMENT or fall back to a generic project experiment
EXPERIMENT_NAME = os.environ.get("EV_MLFLOW_EXPERIMENT", "ev-charging-regional-models")
MODEL_REGISTRY_PREFIX = "ev_carbon_intensity"

mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()
print(f"Experiment: {EXPERIMENT_NAME}")

In [ ]:
# Load metrics CSV produced by notebook 08
metrics_df = pd.read_csv(METRICS_CSV)
print(f"Loaded {len(metrics_df)} rows of metrics")
print(metrics_df.head())

In [ ]:
# Discover all saved .pkl model files
pkl_files = sorted([f for f in os.listdir(MODELS_VOLUME) if f.endswith(".pkl")])
print(f"Found {len(pkl_files)} model files:")
for f in pkl_files:
    print(f"  {f}")

In [ ]:
# Filename pattern: lgbm_p{alpha_int}_{safe_region}.pkl
# e.g. lgbm_p10_south_west_england.pkl -> alpha=0.10, region=south_west_england

ALPHA_MAP = {10: 0.10, 50: 0.50, 90: 0.90}
QUANTILE_LABEL = {0.10: "P10", 0.50: "P50", 0.90: "P90"}

def parse_filename(fname):
    """Returns (alpha_int, alpha_float, safe_region) or None if no match."""
    m = re.match(r"lgbm_p(\d+)_(.+)\.pkl", fname)
    if not m:
        return None
    alpha_int = int(m.group(1))
    safe_region = m.group(2)
    return alpha_int, ALPHA_MAP.get(alpha_int), safe_region

def get_metrics(metrics_df, safe_region, alpha_float):
    """Look up MAE + pinball loss for this model from the metrics CSV."""
    # safe_region uses underscores; CSV 'region' column uses original names
    # Match on alpha first, then filter by comparing safe versions
    subset = metrics_df[metrics_df["alpha"] == alpha_float].copy()
    subset["safe"] = subset["region"].str.lower().str.replace(r"[^a-z0-9]", "_", regex=True)
    row = subset[subset["safe"] == safe_region]
    if row.empty:
        return {}
    return {
        "mae": float(row.iloc[0]["mae"]),
        "pinball_loss": float(row.iloc[0]["pinball_loss"]),
        "region_id": int(row.iloc[0]["region_id"]),
    }

In [ ]:
# Log every model to MLflow and register in the Model Registry
registered = []

for fname in pkl_files:
    parsed = parse_filename(fname)
    if parsed is None:
        print(f"Skipping {fname} — unexpected filename format")
        continue

    alpha_int, alpha_float, safe_region = parsed
    if alpha_float is None:
        print(f"Skipping {fname} — unexpected quantile '{alpha_int}' in filename")
        continue

    quantile = QUANTILE_LABEL[alpha_float]
    region_display = safe_region.replace("_", " ").title()
    model_name = f"{MODEL_REGISTRY_PREFIX}_{safe_region}_{quantile.lower()}"

    model = joblib.load(f"{MODELS_VOLUME}/{fname}")
    metrics = get_metrics(metrics_df, safe_region, alpha_float)

    with mlflow.start_run(run_name=f"{region_display} {quantile}"):
        # Tags
        mlflow.set_tags({
            "region": region_display,
            "quantile": quantile,
            "model_type": "LightGBM",
            "features": "lag_1,lag_2,lag_48,lag_336,rolling_avg_7day,temperature,wind_speed,radiation,temp_roll_avg_7day",
            "training_data": "gold_carbon_weather_regional 2025",
            "notebook": "08_train_regional_models_with_weather",
        })

        # Params
        mlflow.log_params({
            "alpha": alpha_float,
            "region": region_display,
            "safe_region": safe_region,
        })
        if "region_id" in metrics:
            mlflow.log_param("region_id", metrics["region_id"])

        # Metrics
        if metrics:
            mlflow.log_metric("mae", metrics["mae"])
            mlflow.log_metric("pinball_loss", metrics["pinball_loss"])

        # Log model + register
        mlflow.lightgbm.log_model(
            model,
            artifact_path="model",
            registered_model_name=model_name,
        )

        run_id = mlflow.active_run().info.run_id
        registered.append({"model": model_name, "run_id": run_id, "metrics": metrics})
        print(f"Logged: {model_name}  |  MAE={metrics.get('mae', float('nan')):.3f}  |  pinball={metrics.get('pinball_loss', float('nan')):.3f}")

print(f"\nDone — {len(registered)} models logged and registered.")

In [ ]:
# Summary: list all registered models
print(f"{'Model':<55} {'MAE':>8} {'Pinball':>10}")
print("-" * 75)
for r in sorted(registered, key=lambda x: x["model"]):
    mae = r["metrics"].get("mae", float("nan"))
    pb  = r["metrics"].get("pinball_loss", float("nan"))
    print(f"{r['model']:<55} {mae:>8.3f} {pb:>10.3f}")